# 7. Leak-Free Downstream Evaluation

**Pipeline per fold (no data leakage):**
1. Split -> train / test (from `Split_seed_0..9`)
2. Batch correction: fit on **train only** -> transform both
3. PCA: fit on **train only** -> transform both
4. ML model: train on train -> predict on test -> metrics

**Embeddings compared:**
- COMPASS_PT (4224-dim) -- from notebook 3c
- Geneformer (768-dim) -- raw, from harmony files
- Geneformer + Harmony (768-dim) -- Harmony-corrected

**Correction methods (applied inside each fold):**
- none (raw embeddings)
- Adv2-cVAE (adversarial batch correction, fit on train)

**Models:** LogReg + LGBM (with inner GridSearchCV)

**Baseline:** DummyClassifier (majority class)


## 0. Environment Setup

In [ ]:
# !rm -rf batchcor-rna-embeds/
!git clone -b feat/scvi-batch-correction https://github.com/onion-42/batchcor-rna-embeds.git 2>/dev/null || (cd batchcor-rna-embeds && git pull origin feat/scvi-batch-correction)
%cd batchcor-rna-embeds

!pip install -q uv
!uv pip install --system -e .
!pip install -q lightgbm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from loguru import logger

from batchcor_rna_emb.logging_config import set_logger
from batchcor_rna_emb.config import (
    BATCH_COL, SEED,
    COMPASS_PT_EMBEDDING_KEY, SCVI_LATENT_DIM,
)
from batchcor_rna_emb.data_io import load_cohort
from batchcor_rna_emb.modeling.fold_runner import FoldConfig, run_experiment
from batchcor_rna_emb.modeling.plotting import (
    plot_pipeline_comparison, plot_all_results,
)
from batchcor_rna_emb.survival_targets import add_survival_targets
from batchcor_rna_emb.split_utils import generate_stratified_splits

set_logger(level='INFO')
logger.info('Imports OK')


## 1. Configuration

In [ ]:
DATA_INTERIM_DIR = Path('/content/drive/MyDrive/data/interim')
DATA_HARMONY_DIR = Path('/content/drive/MyDrive/data/harmony_embeds')
RESULTS_DIR = Path('/content/drive/MyDrive/data/results')
FIGURES_DIR = RESULTS_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# --- Embedding keys ---
GENEFORMER_RAW_KEY = 'FM_Geneformer_embedding'
GENEFORMER_HARMONY_KEY = 'FM_Geneformer_embedding_Harmony'

TARGETS = ['Response']  # Add 'Target_OS_bin', 'Target_PFS_bin' later
MODELS = ['logreg', 'lgbm']  # Add 'tabpfn' if installed
N_SPLITS = 10
CVAE_EPOCHS = 100

# --- Cohort definitions ---
# Each cohort: (cohort_label, compass_file_dir, compass_filename, harmony_filename)
COHORT_DEFS = [
    {
        'label': 'KIRC_ICI',
        'compass_dir': DATA_INTERIM_DIR,
        'compass_file': 'KIRC_Tissue_ICI_Pred',
        'harmony_dir': DATA_HARMONY_DIR,
        'harmony_file': 'KIRC_Tissue_ICI_Pred_harmony_corrected',
    },
    {
        'label': 'BRCA_SCANB',
        'compass_dir': DATA_INTERIM_DIR,
        'compass_file': 'PUB_BRCA_SCANB',
        'harmony_dir': DATA_HARMONY_DIR,
        'harmony_file': 'PUB_BRCA_SCANB_harmony_corrected',
    },
]

logger.info('Configuration ready')


## 2. Load & Merge Embeddings
For each cohort, load COMPASS (from `data/interim`) and Geneformer/Harmony
(from `data/harmony_embeds`), merge embeddings into a single AnnData.
Generate splits if not present.

In [ ]:
cohort_adatas = {}  # label -> adata

for cdef in COHORT_DEFS:
    label = cdef['label']
    logger.info('\n' + '=' * 70)
    logger.info('Loading cohort: {}', label)
    
    # Load COMPASS adata (has embeddings + splits from 3c/3e)
    compass_path = cdef['compass_dir'] / f"{cdef['compass_file']}.adata.zarr"
    if compass_path.exists():
        adata = load_cohort(compass_path)
        logger.info('  COMPASS loaded: {} samples, obsm keys: {}', adata.n_obs, list(adata.obsm.keys()))
    else:
        logger.warning('  COMPASS file not found: {}, creating empty adata', compass_path)
        adata = None
    
    # Load Harmony adata (has Geneformer embeddings)
    harmony_path = cdef['harmony_dir'] / f"{cdef['harmony_file']}.adata.zarr"
    if harmony_path.exists():
        adata_harm = load_cohort(harmony_path)
        logger.info('  Harmony loaded: {} samples, obsm keys: {}', adata_harm.n_obs, list(adata_harm.obsm.keys()))
        
        if adata is None:
            # No COMPASS file — use harmony as base
            adata = adata_harm
        else:
            # Merge Geneformer embeddings into COMPASS adata by matching obs index
            common_idx = adata.obs_names.intersection(adata_harm.obs_names)
            logger.info('  Common samples: {} / {} (COMPASS) / {} (Harmony)', len(common_idx), adata.n_obs, adata_harm.n_obs)
            
            if len(common_idx) > 0:
                harm_reindexed = adata_harm[common_idx]
                for key in [GENEFORMER_RAW_KEY, GENEFORMER_HARMONY_KEY]:
                    if key in harm_reindexed.obsm:
                        adata.obsm[key] = harm_reindexed.obsm[key].copy()
                        logger.info('  Merged obsm[\'{}\'] shape: {}', key, adata.obsm[key].shape)
            else:
                logger.warning('  No common samples between COMPASS and Harmony!')
                # Fallback: use harmony adata directly
                adata = adata_harm
        del adata_harm
    else:
        logger.warning('  Harmony file not found: {}', harmony_path)
    
    if adata is None:
        logger.error('  No data loaded for {}!', label)
        continue
    
    # Ensure splits exist
    if 'Split_seed_0' not in adata.obs.columns:
        logger.info('  Generating stratified splits...')
        add_survival_targets(adata)
        stratify_cols = [c for c in ['Diagnosis', BATCH_COL, 'Response'] if c in adata.obs.columns]
        generate_stratified_splits(adata, stratify_cols=stratify_cols, n_splits=N_SPLITS, test_size=0.2)
        logger.info('  Splits generated: {}', [c for c in adata.obs.columns if c.startswith('Split_seed')])
    
    cohort_adatas[label] = adata
    logger.info('  Final obsm keys: {}', list(adata.obsm.keys()))
    logger.info('  Final obs columns (splits): {}', [c for c in adata.obs.columns if c.startswith('Split')])

logger.info('\nLoaded cohorts: {}', list(cohort_adatas.keys()))


## 3. Run Experiments
For each cohort x target x pipeline, run all folds x all models.
Pipelines are built dynamically based on available embeddings.

In [ ]:
all_dfs = []

for cohort_label, adata in cohort_adatas.items():
    # Build pipelines dynamically based on available obsm keys
    pipelines = []
    
    # COMPASS pipelines
    if COMPASS_PT_EMBEDDING_KEY in adata.obsm:
        pipelines.append((COMPASS_PT_EMBEDDING_KEY, 'COMPASS_PT', 'none', ''))
        pipelines.append((COMPASS_PT_EMBEDDING_KEY, 'COMPASS_PT', 'cvae_adv2', 'Adv2-cVAE'))
    
    # Geneformer pipelines
    if GENEFORMER_RAW_KEY in adata.obsm:
        pipelines.append((GENEFORMER_RAW_KEY, 'Geneformer', 'none', ''))
    
    # Geneformer + Harmony (correction already applied, no further correction needed)
    if GENEFORMER_HARMONY_KEY in adata.obsm:
        pipelines.append((GENEFORMER_HARMONY_KEY, 'Geneformer+Harmony', 'none', ''))
    
    logger.info('\nCohort: {} | Pipelines: {}', cohort_label, [(p[1], p[3] or 'raw') for p in pipelines])
    
    for target_col in TARGETS:
        if target_col not in adata.obs.columns:
            logger.warning("Target '{}' not in {}, skipping", target_col, cohort_label)
            continue
        
        for emb_key, emb_label, corr_method, corr_label in pipelines:
            cfg = FoldConfig(
                embedding_key=emb_key,
                embedding_label=emb_label,
                target_col=target_col,
                correction_method=corr_method,
                correction_label=corr_label,
                n_pca=128,
                cvae_epochs=CVAE_EPOCHS,
                seed=SEED,
            )
            
            logger.info('\n--- {} | {} | {} ---', cohort_label, target_col, cfg.pipeline_label)
            
            df = run_experiment(
                adata, cfg,
                model_names=MODELS,
                n_splits=N_SPLITS,
            )
            df['cohort'] = cohort_label
            all_dfs.append(df)

if all_dfs:
    results = pd.concat(all_dfs, ignore_index=True)
    logger.info('\nTotal results: {} rows', len(results))
    display(results.head(20))
else:
    logger.error('No results generated!')
    results = pd.DataFrame()


## 4. Save Raw Tables
Save ALL per-fold results so you can regenerate plots without re-running experiments.

In [ ]:
if not results.empty:
    results.to_csv(RESULTS_DIR / 'leak_free_per_fold_results.csv', index=False)
    logger.info('Saved per-fold results to {}', RESULTS_DIR / 'leak_free_per_fold_results.csv')
    
    # Summary table: mean +/- std
    metric_cols = ['roc_auc', 'pr_auc', 'f1', 'f1_weighted', 'f1_macro', 'balanced_accuracy']
    group_cols = ['cohort', 'target', 'pipeline', 'model']
    
    existing_metrics = [c for c in metric_cols if c in results.columns]
    summary = results.groupby(group_cols)[existing_metrics].agg(['mean', 'std']).round(4)
    summary.columns = [f'{m}_{s}' for m, s in summary.columns]
    summary = summary.reset_index()
    
    summary.to_csv(RESULTS_DIR / 'leak_free_summary.csv', index=False)
    logger.info('Saved summary to {}', RESULTS_DIR / 'leak_free_summary.csv')
    display(summary)


## 5. Main Figures: F1 weighted
One plot per (Model, Cohort). Bars sorted by mean, swarm overlay,
DummyClassifier baseline, Wilcoxon+Holm brackets between neighbors.

In [ ]:
if not results.empty:
    figs = plot_all_results(
        results,
        metric_col='f1_weighted',
        metric_label='F1 weighted',
        pipeline_col='pipeline',
        split_col='split',
        save_dir=FIGURES_DIR / 'main',
    )
    for fig in figs:
        plt.show()


## 6. Supplement: ROC-AUC
Same format as main figures, but with ROC-AUC.

In [ ]:
if not results.empty:
    figs_roc = plot_all_results(
        results,
        metric_col='roc_auc',
        metric_label='ROC-AUC',
        pipeline_col='pipeline',
        split_col='split',
        save_dir=FIGURES_DIR / 'supplement_roc_auc',
    )
    for fig in figs_roc:
        plt.show()


## 7. Supplement: Cross-Validation Scores
Inner CV best ROC-AUC for LogReg/LGBM (hyperparameter selection quality).

In [ ]:
if not results.empty and 'cv_best_roc_auc' in results.columns:
    cv_results = results.dropna(subset=['cv_best_roc_auc']).copy()
    if not cv_results.empty:
        figs_cv = plot_all_results(
            cv_results,
            metric_col='cv_best_roc_auc',
            metric_label='Inner CV ROC-AUC (best)',
            pipeline_col='pipeline',
            split_col='split',
            save_dir=FIGURES_DIR / 'supplement_cv',
        )
        for fig in figs_cv:
            plt.show()


## 8. Quick Reload (optional)
If you already ran the experiment and just want to re-plot from saved CSV.

In [ ]:
# Uncomment to reload saved results and re-plot:
# results = pd.read_csv(RESULTS_DIR / 'leak_free_per_fold_results.csv')
# figs = plot_all_results(results, metric_col='f1_weighted', metric_label='F1 weighted',
#                         save_dir=FIGURES_DIR / 'main')
# for fig in figs: plt.show()
